# Current Stress Monitoring Model

## Model Overview

This model estimates the user's **current stress level** on a 0–100 scale without requiring the current self-reported stress value as an input.

The model combines the user's historical self-reported stress with wearable-derived behavioral, sleep, and physiological information.

### Inputs

- Historical mean stress
- Daily steps
- Overall sleep score
- Deep sleep duration
- HRV (RMSSD)

### Output

- Estimated **current stress level (0–100)**

During training, the user's current self-reported stress is used as the ground-truth target rather than an input feature.

This model is intended for passive stress monitoring, particularly when a current self-report is unavailable.

In [20]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [21]:
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "daily_model_v1.csv"
)

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "stress_monitoring_multimodal_v1.joblib"
)

print("Data:", DATA_PATH)
print("Model:", MODEL_PATH)

Data: e:\schoolworks\y3s1\DIP\MindWatch-ML\data\processed\daily_model_v1.csv
Model: e:\schoolworks\y3s1\DIP\MindWatch-ML\models\stress_monitoring_multimodal_v1.joblib


In [22]:
df = pd.read_csv(DATA_PATH)

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(
    ["participant_id", "date"]
).reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Participants:", df["participant_id"].nunique())
print(
    "Date range:",
    df["date"].min(),
    "to",
    df["date"].max()
)

df.head()

Dataset shape: (3118, 14)
Participants: 35
Date range: 2025-02-14 00:00:00 to 2025-07-09 00:00:00


,participant_id,date,current_stress,anxiety,avg_rmssd,overall_score,deep_sleep_in_minutes,daily_steps,STRESS_SCORE,next_date,next_stress_observed,days_to_next,next_day_stress,participant_historical_mean
0,1,2025-02-19,15.0,0.0,NaN,NaN,NaN,NaN,NaN,2025-02-20,33.0,1.0,33.0,NaN
1,1,2025-02-20,33.0,0.0,NaN,NaN,NaN,NaN,NaN,2025-02-21,12.0,1.0,12.0,15.00
2,1,2025-02-21,12.0,0.0,NaN,NaN,NaN,3057.0,NaN,2025-02-22,9.0,1.0,9.0,24.00
3,1,2025-02-22,9.0,8.0,NaN,NaN,NaN,3168.0,NaN,2025-02-23,4.0,1.0,4.0,20.00
4,1,2025-02-23,4.0,7.0,69.259534,78.0,61.0,4287.0,58.0,2025-02-24,37.0,1.0,37.0,17.25


In [23]:
FINAL_FEATURES = [
    "participant_historical_mean",
    "daily_steps",
    "overall_score",
    "deep_sleep_in_minutes",
    "avg_rmssd"
]

TARGET = "current_stress"

model_df = (
    df[df[TARGET].notna()]
    .sort_values(["date", "participant_id"])
    .reset_index(drop=True)
)

print("Target:", TARGET)

print("\nFinal features:")
for feature in FINAL_FEATURES:
    print("-", feature)

print("\nUsable samples:", len(model_df))

print("\nMissing values:")
print(model_df[FINAL_FEATURES].isna().sum())

Target: current_stress

Final features:
- participant_historical_mean
- daily_steps
- overall_score
- deep_sleep_in_minutes
- avg_rmssd

Usable samples: 3118

Missing values:
participant_historical_mean      35
daily_steps                     886
overall_score                  1610
deep_sleep_in_minutes          1610
avg_rmssd                      1481
dtype: int64


In [24]:
unique_dates = np.sort(model_df["date"].unique())

split_idx = int(len(unique_dates) * 0.8)
cutoff_date = unique_dates[split_idx]

train_df = model_df[
    model_df["date"] < cutoff_date
].copy()

test_df = model_df[
    model_df["date"] >= cutoff_date
].copy()

X_train = train_df[FINAL_FEATURES]
X_test = test_df[FINAL_FEATURES]

y_train = train_df[TARGET]
y_test = test_df[TARGET]

print(
    "Cutoff date:",
    pd.Timestamp(cutoff_date).strftime("%Y-%m-%d")
)

print("\nTraining samples:", len(train_df))
print("Test samples:", len(test_df))

print(
    "\nTrain period:",
    train_df["date"].min().strftime("%Y-%m-%d"),
    "to",
    train_df["date"].max().strftime("%Y-%m-%d")
)

print(
    "Test period:",
    test_df["date"].min().strftime("%Y-%m-%d"),
    "to",
    test_df["date"].max().strftime("%Y-%m-%d")
)

print(
    "\nOverlapping dates:",
    len(
        set(train_df["date"])
        & set(test_df["date"])
    )
)

Cutoff date: 2025-06-10

Training samples: 2746
Test samples: 372

Train period: 2025-02-14 to 2025-06-09
Test period: 2025-06-10 to 2025-07-09

Overlapping dates: 0


In [25]:
final_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        LinearRegression()
    )
])

final_model.fit(
    X_train,
    y_train
)

y_pred = final_model.predict(X_test)

print("Model training complete.")
print("Predictions generated:", len(y_pred))

Model training complete.
Predictions generated: 372


In [26]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

r2 = r2_score(
    y_test,
    y_pred
)

print("Current Stress Monitoring Performance")
print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²:   {r2:.3f}")

Current Stress Monitoring Performance
MAE:  16.789
RMSE: 22.126
R²:   0.237


In [27]:
X_all = model_df[FINAL_FEATURES]
y_all = model_df[TARGET]

deployment_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        LinearRegression()
    )
])

deployment_model.fit(
    X_all,
    y_all
)

print("Deployment model trained.")
print("Training samples:", len(X_all))

Deployment model trained.
Training samples: 3118


In [28]:
MODEL_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    deployment_model,
    MODEL_PATH
)

print("Model saved to:")
print(MODEL_PATH)

print("\nFile exists:", MODEL_PATH.exists())

Model saved to:
e:\schoolworks\y3s1\DIP\MindWatch-ML\models\stress_monitoring_multimodal_v1.joblib

File exists: True
